# AnimeGANv3 — Video to Animation on Colab T4
Select a **T4 GPU**, then **Runtime → Run all**. Upload one local video, compare three short style/speed previews, then render the selected stronger animation look. The default full render uses 960 max-edge inference and fast H.264 output.


In [ ]:
import os, subprocess, sys
REPO = '/content/video-to-animation'
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git','fetch','origin','animeganv3-baseline'],cwd=REPO,check=True)
    subprocess.run(['git','reset','--hard','origin/animeganv3-baseline'],cwd=REPO,check=True)
else:
    subprocess.run(['git','clone','-b','animeganv3-baseline','https://github.com/v-tech-hub/video-to-animation.git',REPO],check=True)
os.chdir(REPO)
print('Working directory:', os.getcwd())
subprocess.run(['git','log','-1','--oneline'],check=True)
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip uninstall -y -q onnxruntime onnxruntime-gpu || true
!pip install -q -r requirements-colab.txt


## Input video
Upload one local video file. The notebook normalizes it to `/content/input.<ext>`, validates the video stream with ffprobe, benchmarks the first 120 frames, then renders the same file.


In [ ]:
from pathlib import Path
import shutil
import subprocess
from google.colab import files

print('Upload input video...')
uploaded = files.upload()
videos = [
    name for name in uploaded
    if Path(name).suffix.lower() in {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
]
assert videos, 'Upload a supported video file.'

src = Path(videos[0])
VIDEO = '/content/input' + src.suffix.lower()
shutil.copyfile(src, VIDEO)

assert Path(VIDEO).exists() and Path(VIDEO).stat().st_size > 0, 'Input video was not created.'
print('Video:', VIDEO, f'({Path(VIDEO).stat().st_size / 1024 / 1024:.2f} MB)')

probe = subprocess.run([
    'ffprobe', '-v', 'error', '-select_streams', 'v:0',
    '-show_entries', 'stream=codec_name,width,height,r_frame_rate,duration',
    '-of', 'default=noprint_wrappers=1', VIDEO
], text=True, capture_output=True)
print(probe.stdout)
assert probe.returncode == 0 and probe.stdout.strip(), probe.stderr or 'ffprobe could not read the uploaded video.'


## Style + speed preview
Render the first 180 frames into three small H.264 previews. Compare the current look against stronger cartoon post-processing and the faster 960 max-edge setting before spending time on the full video.


In [ ]:
import os, time, subprocess
from pathlib import Path
import urllib.request
from google.colab import files

MODEL = '/content/AnimeGANv3_Hayao_36.onnx'
MODEL_URL = 'https://github.com/TachibanaYoshino/AnimeGANv3/releases/download/v1.1.0/AnimeGANv3_Hayao_36.onnx'
if not Path(MODEL).exists():
    print('Downloading AnimeGANv3 Hayao model...')
    urllib.request.urlretrieve(MODEL_URL, MODEL)

BEST_DEVICE = 'gpu'
PREVIEW_FRAMES = 180
PREVIEW_VARIANTS = [
    ('base-1024',   'base',   1024),
    ('strong-1024', 'strong', 1024),
    ('strong-960',  'strong',  960),
]
script = os.path.join(REPO, 'tools', 'video2anime.py')
code = Path(script).read_text()
assert '--style' in code and '--max-frames' in code, 'Old renderer detected; rerun the setup cell.'

preview_dir = '/content/style-previews'
os.makedirs(preview_dir, exist_ok=True)
preview_results = []

for label, style, edge in PREVIEW_VARIANTS:
    print(f'\n=== Preview: {label} ===', flush=True)
    cmd = [
        'python', '-u', script,
        '-i', VIDEO, '-o', preview_dir, '-m', MODEL, '-d', BEST_DEVICE,
        '--style', style, '--max-edge', str(edge),
        '--output-max-edge', '960',
        '--preset', 'ultrafast', '--crf', '22',
        '--max-frames', str(PREVIEW_FRAMES),
    ]
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed_preview = time.perf_counter() - t0
    print(proc.stdout)
    assert proc.returncode == 0, f'Preview failed: {label}'
    candidates = sorted(Path(preview_dir).glob(f'*_{style}_{edge}.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
    assert candidates, f'Preview output missing: {label}'
    preview_results.append((label, candidates[0], elapsed_preview))
    print(f'{label}: {elapsed_preview:.2f}s')

print('\n=== PREVIEW SUMMARY ===')
for label, path, seconds in preview_results:
    print(f'{label:>12}: {seconds:7.2f}s  -> {path}')

# Download all three short previews so the visual choice is made before the full render.
for _, path, _ in preview_results:
    files.download(str(path))

# Default full render: stronger animation look + faster 960 inference.
RENDER_STYLE = 'strong'
RENDER_MAX_EDGE = 960
# Keep source dimensions for final delivery. Set to 960 if speed matters more than 1080p output.
OUTPUT_MAX_EDGE = 0
ENCODE_PRESET = 'ultrafast'
print(f'\nSelected full render: style={RENDER_STYLE}, max_edge={RENDER_MAX_EDGE}, output_max_edge={OUTPUT_MAX_EDGE or "source"}')


## Full render
After checking the previews, change `RENDER_STYLE` or `RENDER_MAX_EDGE` above if needed. The default is the stronger cartoon treatment at max edge 960, with fast H.264 encoding.


In [ ]:
import time, threading
os.chdir(REPO)
os.makedirs('output', exist_ok=True)

cmd = [
    'python', '-u', script,
    '-i', VIDEO, '-o', os.path.join(REPO, 'output'), '-m', MODEL, '-d', BEST_DEVICE,
    '--style', RENDER_STYLE,
    '--max-edge', str(RENDER_MAX_EDGE),
    '--output-max-edge', str(OUTPUT_MAX_EDGE),
    '--preset', ENCODE_PRESET,
    '--crf', '21',
]
print('=== Full AnimeGANv3 render ===', flush=True)
print('Command:', ' '.join(cmd), flush=True)

stop_gpu = threading.Event()
def gpu_monitor():
    while not stop_gpu.wait(10):
        q = subprocess.run(
            ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,power.draw', '--format=csv,noheader,nounits'],
            capture_output=True, text=True
        )
        if q.returncode == 0:
            print('[gpu] util%, memMiB, powerW:', q.stdout.strip(), flush=True)

threading.Thread(target=gpu_monitor, daemon=True).start()
start = time.perf_counter()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
returncode = proc.wait()
stop_gpu.set()
elapsed = time.perf_counter() - start
print(f'\nExit code: {returncode}')
print(f'Wall time: {elapsed:.2f} s')
if returncode != 0:
    raise RuntimeError(f'AnimeGANv3 failed with exit code {returncode}')


## Validate and download
Verify the rendered MP4 is H.264, print timing, then download it.


In [ ]:
import cv2
import subprocess
from pathlib import Path
from google.colab import files

cap = cv2.VideoCapture(VIDEO)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
duration = frames / fps if fps else 0
render_fps = frames / elapsed if elapsed else 0
rtf = elapsed / duration if duration else 0
print(f'Frames: {frames}')
print(f'Source FPS: {fps:.3f}')
print(f'Source duration: {duration:.2f} s')
print(f'Render throughput: {render_fps:.2f} FPS')
print(f'Realtime factor: {rtf:.2f}x')

pattern = f'*_{RENDER_STYLE}_{RENDER_MAX_EDGE}*.mp4'
outs = sorted(Path(REPO, 'output').glob(pattern), key=lambda x: x.stat().st_mtime, reverse=True)
assert outs, f'No rendered MP4 found for {pattern}'
final_output = outs[0]
probe = subprocess.run([
    'ffprobe', '-v', 'error', '-select_streams', 'v:0',
    '-show_entries', 'stream=codec_name,pix_fmt,width,height,r_frame_rate',
    '-of', 'default=noprint_wrappers=1', str(final_output)
], text=True, capture_output=True)
print('Output:', final_output)
print(probe.stdout)
assert probe.returncode == 0 and 'codec_name=h264' in probe.stdout and 'pix_fmt=yuv420p' in probe.stdout, probe.stderr or 'Output is not QuickTime-compatible H.264.'
files.download(str(final_output))
